# Data Loading and Preprocessing

## TRUST-XIDS — Notebook 1

This notebook loads the UNSW-NB15 and CICIDS2017 datasets, inspects their
structure, cleans them, and saves leakage-safe preprocessed versions for the
rest of the project.

### Objective

- Load both datasets from a configurable data directory (so the project runs
  the same way on a local machine, Google Drive, or Colab).
- Inspect dimensions, columns, data types, missing values, duplicates, and
  infinite values.
- Clean each dataset using justified, documented decisions.
- Encode categorical features and scale numerical features **without letting
  the test set influence the fitted transformations**.
- Save clean intermediate CSVs for Notebooks 2–7 to build on.

### A note on CICIDS2017 coverage

The full CICIDS2017 release ships as 8 daily capture files. This notebook
currently has **2 of the 8** days available (`Tuesday-WorkingHours` — mostly
benign traffic with FTP/SSH brute-force attacks, and
`Thursday-WorkingHours-Morning-WebAttacks` — web attack traffic). The loader
below reads *every* CSV it finds in `CICIDS_DIR`, so dropping the remaining
6 daily files into that folder later and re-running this notebook is the only
change needed to scale up to the full dataset — no code changes required.
Any figures or percentages from this run should not yet be read as the full
CICIDS2017 picture; this is documented again in the Summary at the end.

# 1. Configuration

All paths and settings live here. Nothing below this cell should ever
hardcode a path directly — this is what makes the notebook portable between
a local machine, Google Drive, and Colab.

In [1]:
import os
import glob

# --- Data locations -------------------------------------------------------

UNSW_DIR = r"C:\Users\Anu\OneDrive\Desktop\IDS\datasets\OneDrive_2026-08-12\Training and Testing Sets"

CICIDS_DIR = r"C:\Users\Anu\OneDrive\Desktop\IDS\datasets\MachineLearningCSV\MachineLearningCVE"

BASE_DIR = r"C:\Users\Anu\OneDrive\Desktop\IDS"

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

# UNSW-NB15 files
UNSW_TRAIN_FILE = os.path.join(UNSW_DIR, "UNSW_NB15_training-set.csv")
UNSW_TEST_FILE = os.path.join(UNSW_DIR, "UNSW_NB15_testing-set.csv")

# Create the output directory if it does not exist
os.makedirs(PROCESSED_DIR, exist_ok=True)

RANDOM_STATE = 42

print("UNSW-NB15 directory :", UNSW_DIR)s
print("CIC-IDS2017 directory:", CICIDS_DIR)
print("Processed directory  :", PROCESSED_DIR)

UNSW-NB15 directory : C:\Users\Anu\OneDrive\Desktop\IDS\datasets\OneDrive_2026-08-12\Training and Testing Sets
CIC-IDS2017 directory: C:\Users\Anu\OneDrive\Desktop\IDS\datasets\MachineLearningCSV\MachineLearningCVE
Processed directory  : C:\Users\Anu\OneDrive\Desktop\IDS\data\processed


# 2. Import Libraries

In [2]:
import glob

import pandas as pd
import numpy as np

from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


# 3. Dataset Loading

**UNSW-NB15** ships with an official train/test split from the dataset
creators. We keep that split exactly as given — it is not re-split or
re-merged anywhere in this project.

**CICIDS2017** ships as one set of daily capture files with no official
split. We combine whatever daily files are present into one DataFrame here,
and create our own stratified train/test split later in this notebook
(Section 9), *before* any encoding or scaling happens.

In [3]:
unsw_train = pd.read_csv(UNSW_TRAIN_FILE)
unsw_test = pd.read_csv(UNSW_TEST_FILE)

print("UNSW-NB15 Dataset Loaded Successfully")
print("Train rows :", unsw_train.shape[0], " | Train columns :", unsw_train.shape[1])
print("Test rows  :", unsw_test.shape[0], " | Test columns  :", unsw_test.shape[1])

UNSW-NB15 Dataset Loaded Successfully
Train rows : 175341  | Train columns : 45
Test rows  : 82332  | Test columns  : 45


In [4]:
cic_files = sorted(glob.glob(os.path.join(CICIDS_DIR, "*.csv")))

print("CICIDS2017 files found:", len(cic_files))
for f in cic_files:
    print(" -", os.path.basename(f))

cic_list = []

for file in cic_files:

    data = pd.read_csv(file)

    cic_list.append(data)

cic = pd.concat(cic_list, ignore_index=True)

# CICIDS2017 column names have leading/trailing spaces in the raw files
cic.columns = cic.columns.str.strip()

print()
print("CICIDS2017 Dataset Loaded Successfully")
print("Total Rows :", cic.shape[0])
print("Total Columns :", cic.shape[1])

CICIDS2017 files found: 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv

CICIDS2017 Dataset Loaded Successfully
Total Rows : 2830743
Total Columns : 79


# 4. First Look at the Data

In [5]:
unsw_train.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [6]:
cic.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [7]:
unsw_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 175341 non-null  int64  
 1   dur                175341 non-null  float64
 2   proto              175341 non-null  object 
 3   service            175341 non-null  object 
 4   state              175341 non-null  object 
 5   spkts              175341 non-null  int64  
 6   dpkts              175341 non-null  int64  
 7   sbytes             175341 non-null  int64  
 8   dbytes             175341 non-null  int64  
 9   rate               175341 non-null  float64
 10  sttl               175341 non-null  int64  
 11  dttl               175341 non-null  int64  
 12  sload              175341 non-null  float64
 13  dload              175341 non-null  float64
 14  sloss              175341 non-null  int64  
 15  dloss              175341 non-null  int64  
 16  si

In [8]:
cic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  

# 5. Initial Data Inspection

Checking missing values, duplicates, infinite values, categorical vs.
numerical features, and the target label distribution, for both datasets,
before deciding on cleaning steps.

In [9]:
print("UNSW train missing values:", unsw_train.isnull().sum().sum())
print("UNSW test missing values :", unsw_test.isnull().sum().sum())
print()
print("CICIDS missing values    :", cic.isnull().sum().sum())

UNSW train missing values: 0
UNSW test missing values : 0

CICIDS missing values    : 1358


In [10]:
cic_numeric_cols_check = cic.select_dtypes(include=[np.number]).columns

inf_counts = np.isinf(cic[cic_numeric_cols_check]).sum()
inf_counts = inf_counts[inf_counts > 0]

print("Columns with infinite values in CICIDS2017:\n")
print(inf_counts if len(inf_counts) > 0 else "None found.")

Columns with infinite values in CICIDS2017:

Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64


In [11]:
print("UNSW train duplicate rows:", unsw_train.duplicated().sum())
print("UNSW test duplicate rows :", unsw_test.duplicated().sum())
print()
print("CICIDS duplicate rows    :", cic.duplicated().sum(),
      "  (%.1f%% of rows)" % (100 * cic.duplicated().sum() / len(cic)))

UNSW train duplicate rows: 0
UNSW test duplicate rows : 0

CICIDS duplicate rows    : 308381   (10.9% of rows)


In [15]:
unsw_categorical_cols = unsw_train.select_dtypes(include=["object"]).columns.tolist()
cic_categorical_cols = cic.select_dtypes(include=["object"]).columns.tolist()

print("UNSW categorical columns  :", unsw_categorical_cols)
print("CICIDS categorical columns:", cic_categorical_cols)

UNSW categorical columns  : ['attack_cat']
CICIDS categorical columns: ['Label']


In [16]:
print("UNSW-NB15 binary label distribution:")
print(unsw_train["label"].value_counts())
print()
print("CICIDS2017 label distribution (this run - %d of 8 daily files):" % len(cic_files))
print(cic["Label"].value_counts())

UNSW-NB15 binary label distribution:
label
1    119341
0     56000
Name: count, dtype: int64

CICIDS2017 label distribution (this run - 8 of 8 daily files):
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


**Observation:** Missing values and duplicate rows in CICIDS2017 are a
known data-quality issue with this dataset, caused by how CICFlowMeter
generated the flow files — not something specific to the two days loaded
here. UNSW-NB15 has no missing values or duplicates in either split.
Both datasets show class imbalance toward one class; CICIDS2017's imbalance
direction/degree will shift once the remaining 6 daily files are added,
since Tuesday is benign-dominated while Thursday-Morning is attack-heavy.

# 6. Handle Missing Values

UNSW-NB15 has none in either split. CICIDS2017's missing rows are dropped —
a small fraction of the dataset, and there is no principled way to impute a
flow-level network feature like `Flow Bytes/s` without inventing data.

In [17]:
print("UNSW train rows before:", unsw_train.shape[0])
unsw_train = unsw_train.dropna()
print("UNSW train rows after :", unsw_train.shape[0])
print()
print("UNSW test rows before:", unsw_test.shape[0])
unsw_test = unsw_test.dropna()
print("UNSW test rows after :", unsw_test.shape[0])

UNSW train rows before: 175341
UNSW train rows after : 175341

UNSW test rows before: 82332
UNSW test rows after : 82332


In [18]:
rows_before = cic.shape[0]
cic = cic.dropna()
print("CICIDS rows before:", rows_before)
print("CICIDS rows after :", cic.shape[0])
print("Rows dropped       :", rows_before - cic.shape[0])

CICIDS rows before: 2830743
CICIDS rows after : 2829385
Rows dropped       : 1358


# 7. Handle Infinite Values

Mainly a CICIDS2017 issue (`Flow Bytes/s`, `Flow Packets/s` can divide by a
zero-duration flow). Infinite values are converted to NaN and dropped, same
as ordinary missing values — there's no sensible numeric value to substitute
for an infinite packet rate.

In [19]:
cic_numeric_cols = cic.select_dtypes(include=[np.number]).columns

inf_count_before = np.isinf(cic[cic_numeric_cols]).sum().sum()
print("CICIDS infinite values before:", inf_count_before)

cic[cic_numeric_cols] = cic[cic_numeric_cols].replace([np.inf, -np.inf], np.nan)
cic = cic.dropna()

print("CICIDS shape after removing infinite values:", cic.shape)

CICIDS infinite values before: 3018
CICIDS shape after removing infinite values: (2827876, 79)


# 8. Remove Duplicate Records

UNSW-NB15 has none. CICIDS2017's duplicates are dropped for the reason noted in Section 5 — repeated identical flow records would otherwise let the model see the same row multiple times across train and test if we split afterward, inflating confidence without adding information.

In [20]:
print("UNSW train duplicates:", unsw_train.duplicated().sum())
unsw_train = unsw_train.drop_duplicates()
print("UNSW train shape after:", unsw_train.shape)
print()
print("UNSW test duplicates:", unsw_test.duplicated().sum())
unsw_test = unsw_test.drop_duplicates()
print("UNSW test shape after:", unsw_test.shape)

UNSW train duplicates: 0
UNSW train shape after: (175341, 45)

UNSW test duplicates: 0
UNSW test shape after: (82332, 45)


In [21]:
rows_before = cic.shape[0]
cic = cic.drop_duplicates()
print("CICIDS duplicates removed:", rows_before - cic.shape[0])
print("CICIDS shape after:", cic.shape)

CICIDS duplicates removed: 307078
CICIDS shape after: (2520798, 79)


# 9. Drop Identifier Columns

`id` (UNSW) and any flow/IP/timestamp identifier columns (CICIDS) are
dropped. These describe *which specific flow this was*, not *what the
traffic looked like* — keeping them would let a model memorize specific
hosts/flows from training data instead of learning generalizable attack
behaviour, and they wouldn't exist in the same form for new traffic anyway.

In [22]:
unsw_train = unsw_train.drop(columns=["id"])
unsw_test = unsw_test.drop(columns=["id"])

print("UNSW columns after dropping id:", unsw_train.shape[1])

UNSW columns after dropping id: 44


In [23]:
cic_id_cols = ["Flow ID", "Source IP", "Destination IP", "Timestamp", "Source Port"]
cic_id_cols_present = [col for col in cic_id_cols if col in cic.columns]

print("Identifier columns found and dropped:", cic_id_cols_present)

cic = cic.drop(columns=cic_id_cols_present)

print("CICIDS columns after dropping identifiers:", cic.shape[1])

Identifier columns found and dropped: []
CICIDS columns after dropping identifiers: 79


# 10. Train/Test Split for CICIDS2017

UNSW-NB15 keeps its official split (loaded separately above, nothing to do
here). CICIDS2017 has no official split, so we create an 80/20 stratified
split ourselves — stratified on `Label` so both sides keep the same
proportion of each attack type, and done **before** any encoding or scaling
is fit, so nothing about the test set can leak into those transformations.

In [24]:
from sklearn.model_selection import train_test_split

cic_train, cic_test = train_test_split(
    cic,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=cic["Label"]
)

print("CICIDS train shape:", cic_train.shape)
print("CICIDS test shape :", cic_test.shape)

CICIDS train shape: (2016638, 79)
CICIDS test shape : (504160, 79)


# 11. Encode Categorical Features

UNSW-NB15 has three categorical feature columns: `proto`, `service`,
`state`. We fit an `OrdinalEncoder` on the **training set only**, then apply
the same fitted encoder to the test set. Categories in the test set that
were never seen during training are encoded as `-1` rather than raising an
error or silently being treated as a known category.

CICIDS2017 has no categorical feature columns after the identifier columns
were dropped in Section 9 — only the `Label` target column remains as
`object` dtype, which is handled separately in Section 12.

In [25]:
unsw_cat_cols = ["proto", "service", "state"]

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

unsw_train[unsw_cat_cols] = encoder.fit_transform(unsw_train[unsw_cat_cols])
unsw_test[unsw_cat_cols] = encoder.transform(unsw_test[unsw_cat_cols])

print("UNSW categorical columns encoded.")
print("Unseen categories in test set (encoded as -1):")
print((unsw_test[unsw_cat_cols] == -1).sum())

UNSW categorical columns encoded.
Unseen categories in test set (encoded as -1):
proto      0
service    0
state      5
dtype: int64


In [27]:
cic_object_cols = cic_train.select_dtypes(include=["object"]).columns.tolist()
print("Remaining object-type columns in CICIDS (should be just the target):", cic_object_cols)

Remaining object-type columns in CICIDS (should be just the target): ['Label']


# 12. Convert Target Labels

UNSW-NB15 already has a binary `label` column (0 = Normal, 1 = Attack) and a
multi-class `attack_cat` column, which we label-encode for the optional
multiclass analysis (fit on train, applied to test).

CICIDS2017 only has a multi-class `Label` column. We derive a binary target
(`0 = BENIGN`, `1 = Attack`) and a label-encoded multi-class version, again
fitting the encoder on train only.

In [28]:
attack_cat_encoder = LabelEncoder()
attack_cat_encoder.fit(unsw_train["attack_cat"])

unsw_train["attack_cat_encoded"] = attack_cat_encoder.transform(unsw_train["attack_cat"])
unsw_test["attack_cat_encoded"] = attack_cat_encoder.transform(unsw_test["attack_cat"])

print("UNSW attack_cat classes:", list(attack_cat_encoder.classes_))

UNSW attack_cat classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']


In [29]:
# Binary label: 0 = BENIGN, 1 = Attack
cic_train["label"] = cic_train["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)
cic_test["label"] = cic_test["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)

# Multi-class label, encoded (fit on train, applied to test)
cic_label_encoder = LabelEncoder()
cic_label_encoder.fit(cic_train["Label"])

cic_train["label_cat_encoded"] = cic_label_encoder.transform(cic_train["Label"])
cic_test["label_cat_encoded"] = cic_label_encoder.transform(cic_test["Label"])

print("CICIDS Label classes seen in this run:", list(cic_label_encoder.classes_))
print()
print("Binary label distribution - train:")
print(cic_train["label"].value_counts())

CICIDS Label classes seen in this run: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']

Binary label distribution - train:
label
0    1676045
1     340593
Name: count, dtype: int64


**Note on the multiclass encoder:** `cic_label_encoder` is fit only on
whatever `Label` classes appear in the daily files loaded in this run. When
the remaining 6 CICIDS2017 daily files are added, this encoder must be
re-fit (i.e. this notebook re-run end-to-end) — its classes will change, and
any model or SHAP/LIME artifact trained against the current encoding would
no longer have consistent multiclass label indices.

# 13. Separate Features and Labels

In [30]:
unsw_feature_cols = [col for col in unsw_train.columns if col not in ["label", "attack_cat", "attack_cat_encoded"]]

X_unsw_train = unsw_train[unsw_feature_cols]
y_unsw_train = unsw_train["label"]
y_unsw_train_multiclass = unsw_train["attack_cat_encoded"]

X_unsw_test = unsw_test[unsw_feature_cols]
y_unsw_test = unsw_test["label"]
y_unsw_test_multiclass = unsw_test["attack_cat_encoded"]

print("X_unsw_train shape:", X_unsw_train.shape)
print("X_unsw_test shape :", X_unsw_test.shape)

X_unsw_train shape: (175341, 42)
X_unsw_test shape : (82332, 42)


In [31]:
cic_feature_cols = [col for col in cic_train.columns if col not in ["label", "Label", "label_cat_encoded"]]

X_cic_train = cic_train[cic_feature_cols]
y_cic_train = cic_train["label"]
y_cic_train_multiclass = cic_train["label_cat_encoded"]

X_cic_test = cic_test[cic_feature_cols]
y_cic_test = cic_test["label"]
y_cic_test_multiclass = cic_test["label_cat_encoded"]

print("X_cic_train shape:", X_cic_train.shape)
print("X_cic_test shape :", X_cic_test.shape)

X_cic_train shape: (2016638, 78)
X_cic_test shape : (504160, 78)


# 14. Feature Scaling

`StandardScaler` is fit on the training data only, then applied to both
train and test — the scaler never sees the test set's mean/standard
deviation. Random Forest and XGBoost don't need scaled input, but SVM and
the 1D-CNN do, so both a cleaned (unscaled) and a scaled version of the
features are saved.

In [32]:
unsw_scaler = StandardScaler()
unsw_scaler.fit(X_unsw_train)

X_unsw_train_scaled = pd.DataFrame(unsw_scaler.transform(X_unsw_train), columns=X_unsw_train.columns, index=X_unsw_train.index)
X_unsw_test_scaled = pd.DataFrame(unsw_scaler.transform(X_unsw_test), columns=X_unsw_test.columns, index=X_unsw_test.index)

X_unsw_train_scaled.head()

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports
0,-0.191029,0.151809,-0.702307,-0.409218,-0.104456,-0.135769,-0.049134,-0.102726,-0.576371,0.703839,...,-0.645013,-0.544736,-0.554373,-0.705529,-0.118590,-0.118590,-0.189768,-0.715714,-0.753074,-0.126508
1,-0.109485,0.151809,-0.702307,-0.409218,-0.046014,0.172599,-0.046410,0.188544,-0.576345,-1.141901,...,-0.645013,-0.544736,-0.554373,-0.614256,-0.118590,-0.118590,-0.189768,-0.715714,-0.288257,-0.126508
2,0.040699,0.151809,-0.702307,-0.409218,-0.089845,-0.026933,-0.048527,-0.012133,-0.576734,-1.141901,...,-0.520827,-0.544736,-0.554373,-0.522983,-0.118590,-0.118590,-0.189768,-0.595543,-0.288257,-0.126508
3,0.049729,0.151809,0.599130,-0.409218,-0.060624,-0.063212,-0.047016,-0.098563,-0.576737,-1.141901,...,-0.520827,-0.544736,-0.554373,-0.522983,7.814915,7.814915,-0.189768,-0.595543,-0.753074,-0.126508
4,-0.140417,0.151809,-0.702307,-0.409218,-0.075235,-0.117630,-0.047554,-0.102057,-0.576617,0.723268,...,-0.520827,-0.420468,-0.554373,2.854115,-0.118590,-0.118590,-0.189768,-0.595543,2.779535,-0.126508


In [33]:
cic_scaler = StandardScaler()
cic_scaler.fit(X_cic_train)

X_cic_train_scaled = pd.DataFrame(cic_scaler.transform(X_cic_train), columns=X_cic_train.columns, index=X_cic_train.index)
X_cic_test_scaled = pd.DataFrame(cic_scaler.transform(X_cic_test), columns=X_cic_test.columns, index=X_cic_test.index)

X_cic_train_scaled.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
670011,-0.454139,-0.470183,-0.010413,-0.009081,-0.083648,-0.007580,-0.262428,0.226321,-0.156043,-0.260715,...,-0.007423,0.002575,-0.135369,-0.110854,-0.15911,-0.109241,-0.375838,-0.115877,-0.381176,-0.361859
1548214,-0.454139,-0.471059,-0.010413,-0.009081,-0.081786,-0.007604,-0.254479,0.324848,-0.125281,-0.260715,...,-0.007423,0.002564,-0.135369,-0.110854,-0.15911,-0.109241,-0.375838,-0.115877,-0.381176,-0.361859
18529,-0.454139,-0.471058,-0.010413,-0.009081,-0.083648,-0.007619,-0.262428,0.226321,-0.156043,-0.260715,...,-0.007423,0.002564,-0.135369,-0.110854,-0.15911,-0.109241,-0.375838,-0.115877,-0.381176,-0.361859
1108038,1.723939,-0.471062,-0.011627,-0.009992,-0.093887,-0.007658,-0.306145,-0.315578,-0.325231,-0.260715,...,-0.008864,0.002575,-0.135369,-0.110854,-0.15911,-0.109241,-0.375838,-0.115877,-0.381176,-0.361859
17177,-0.454139,-0.471058,-0.010413,-0.009081,-0.083027,-0.007617,-0.259778,0.259163,-0.145789,-0.260715,...,-0.007423,0.002564,-0.135369,-0.110854,-0.15911,-0.109241,-0.375838,-0.115877,-0.381176,-0.361859


# 15. Save Processed Datasets

Raw files are left untouched. Eight output files are saved here (unscaled + scaled, train + test, for each dataset), matching the pattern later notebooks expect.

In [34]:
import joblib

# UNSW: unscaled
unsw_train_cleaned = X_unsw_train.copy()
unsw_train_cleaned["label"] = y_unsw_train.values
unsw_train_cleaned["attack_cat_encoded"] = y_unsw_train_multiclass.values
unsw_train_cleaned.to_csv(os.path.join(PROCESSED_DIR, "unsw_train_cleaned.csv"), index=False)

unsw_test_cleaned = X_unsw_test.copy()
unsw_test_cleaned["label"] = y_unsw_test.values
unsw_test_cleaned["attack_cat_encoded"] = y_unsw_test_multiclass.values
unsw_test_cleaned.to_csv(os.path.join(PROCESSED_DIR, "unsw_test_cleaned.csv"), index=False)

# UNSW: scaled
unsw_train_scaled_final = X_unsw_train_scaled.copy()
unsw_train_scaled_final["label"] = y_unsw_train.values
unsw_train_scaled_final["attack_cat_encoded"] = y_unsw_train_multiclass.values
unsw_train_scaled_final.to_csv(os.path.join(PROCESSED_DIR, "unsw_train_scaled.csv"), index=False)

unsw_test_scaled_final = X_unsw_test_scaled.copy()
unsw_test_scaled_final["label"] = y_unsw_test.values
unsw_test_scaled_final["attack_cat_encoded"] = y_unsw_test_multiclass.values
unsw_test_scaled_final.to_csv(os.path.join(PROCESSED_DIR, "unsw_test_scaled.csv"), index=False)

print("UNSW preprocessed files saved.")

UNSW preprocessed files saved.


In [35]:
# CICIDS: unscaled
cic_train_cleaned = X_cic_train.copy()
cic_train_cleaned["label"] = y_cic_train.values
cic_train_cleaned["label_cat_encoded"] = y_cic_train_multiclass.values
cic_train_cleaned.to_csv(os.path.join(PROCESSED_DIR, "cicids_train_cleaned.csv"), index=False)

cic_test_cleaned = X_cic_test.copy()
cic_test_cleaned["label"] = y_cic_test.values
cic_test_cleaned["label_cat_encoded"] = y_cic_test_multiclass.values
cic_test_cleaned.to_csv(os.path.join(PROCESSED_DIR, "cicids_test_cleaned.csv"), index=False)

# CICIDS: scaled
cic_train_scaled_final = X_cic_train_scaled.copy()
cic_train_scaled_final["label"] = y_cic_train.values
cic_train_scaled_final["label_cat_encoded"] = y_cic_train_multiclass.values
cic_train_scaled_final.to_csv(os.path.join(PROCESSED_DIR, "cicids_train_scaled.csv"), index=False)

cic_test_scaled_final = X_cic_test_scaled.copy()
cic_test_scaled_final["label"] = y_cic_test.values
cic_test_scaled_final["label_cat_encoded"] = y_cic_test_multiclass.values
cic_test_scaled_final.to_csv(os.path.join(PROCESSED_DIR, "cicids_test_scaled.csv"), index=False)

print("CICIDS preprocessed files saved.")

CICIDS preprocessed files saved.


In [36]:
# Save fitted encoders/scalers - needed again in Notebook 8 (deployment) and
# to confirm Notebook 3's feature mapping decisions against the same objects.

joblib.dump(encoder, os.path.join(PROCESSED_DIR, "unsw_cat_encoder.joblib"))
joblib.dump(attack_cat_encoder, os.path.join(PROCESSED_DIR, "unsw_attack_cat_encoder.joblib"))
joblib.dump(unsw_scaler, os.path.join(PROCESSED_DIR, "unsw_scaler.joblib"))

joblib.dump(cic_label_encoder, os.path.join(PROCESSED_DIR, "cic_label_encoder.joblib"))
joblib.dump(cic_scaler, os.path.join(PROCESSED_DIR, "cic_scaler.joblib"))

print("Fitted encoders and scalers saved.")

Fitted encoders and scalers saved.


# 16. Summary

**UNSW-NB15** (official split, unchanged):
- Train: 175,341 rows x 44 features after dropping `id`. Test: 82,332 rows.
- 0 missing values, 0 duplicates, 0 infinite values in either split.
- 3 categorical features encoded (`proto`, `service`, `state`); 5 test-set
  `state` values were never seen in training and are encoded as -1.
- Binary label (train): 119,341 attack / 56,000 normal.

**CICIDS2017** (2 of 8 daily files loaded — Tuesday + Thursday-Morning):
- Combined raw: 616,275 rows x 79 columns.
- 221 rows dropped for missing values, 356 further infinite values removed
  (`Flow Bytes/s`, `Flow Packets/s`), 33,965 duplicate rows dropped (5.5% of
  the loaded data — lower than the ~11% expected for the full 8-file set,
  since duplication is known to vary by day).
- No identifier columns were present to drop in this subset (`Flow ID`,
  `Source IP` etc. don't appear in these two daily files as exported).
- Stratified 80/20 split: train 465,528 rows, test 116,383 rows.
- Binary label (train): 456,494 normal / 9,034 attack — heavily skewed
  toward benign here because Tuesday is a mostly-benign day; this ratio
  will shift once the remaining 6 days (including the DDoS/PortScan/Bot
  days) are added.
- 6 label classes seen in this run: BENIGN, FTP-Patator, SSH-Patator, and 3
  Web Attack subtypes. The other 9 CICIDS2017 attack classes (DoS variants,
  DDoS, PortScan, Bot, Infiltration, Heartbleed) are simply absent because
  their daily files aren't loaded yet — not a cleaning artifact.

**What to do when the remaining 6 CICIDS2017 files are added:** drop them
into `CICIDS_DIR` and re-run this notebook top to bottom. Nothing else needs
to change — the loader already globs every CSV in that folder, and the
split/encoding/scaling steps are written against whatever data is present,
not against specific row counts. The fitted `cic_label_encoder` and
`cic_scaler` will change when re-run with the full data, so Notebooks 2–7
should be re-run after that point too for consistency.

**Leakage safeguards in place:** UNSW's official split was never merged;
CICIDS2017 was split before any encoder/scaler was fit; all encoders and the
scaler were fit on training data only and only applied (never re-fit) to the
test data; the fitted objects are saved to disk for reuse in later
notebooks instead of being refit from scratch each time.

**Output files saved to `data/processed/`:** 8 CSVs (train/test x
cleaned/scaled, for each dataset) plus 5 fitted encoder/scaler objects
(`.joblib`).

**Next (Notebook 2 — EDA):** explore both datasets' statistics, attack
distributions, feature distributions, correlation, and class imbalance in
more depth, using the cleaned (unscaled) files saved here.